# 03. RAG 검색·답변 품질 검증

02에서 만든 FAISS 인덱스 + LLM(Upstage solar-pro)을 LCEL 체인으로 묶어
시연 질문이 5초 안에 출처와 함께 답변되는지 확인.

**진행 순서**
1. 환경 설정 (API Key + LLM/Embedding 객체)
2. FAISS 인덱스 로드 (`allow_dangerous_deserialization=True`)
3. Retriever 생성 + 검색 동작 빠른 확인
4. 프롬프트 + RAG 체인 (LCEL)
5. 시연 Q1: "PIC 리조트 어때? 가족이 가도 좋아?"
6. 시연 Q2: "5월 둘째 주 괌 날씨 어때?" — RAG 단독 한계 확인 (도구 도입 근거)

## 1. 환경 설정

- `.env`에서 `UPSTAGE_API_KEY` 로드
- LLM: `ChatUpstage(model="solar-pro")` + 가드 4개
- Embedding: `solar-embedding-1-large-query` — 02에서 인덱싱 시 쓴 모델과 **같아야 함** (검색 시 query suffix 자동 적용)

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
api_key = os.getenv("UPSTAGE_API_KEY")
if not api_key:
    raise ValueError(
        "UPSTAGE_API_KEY를 찾을 수 없습니다.\n"
        "프로젝트 루트의 .env에 UPSTAGE_API_KEY=sk-...를 입력하세요."
    )
print(f"API Key 로드 성공: {api_key[:8]}...")

from langchain_upstage import ChatUpstage, UpstageEmbeddings

# LLM 가드 (강의 contract analyzer 8장 패턴)
llm = ChatUpstage(
    model="solar-pro",
    temperature=0,      # 시연·검증 단계에서 같은 질문→같은 답 (재현성)
    max_tokens=1024,    # 답변 길이 가드 (가족 챗봇 답변엔 충분)
    timeout=30,         # 30초 응답 없으면 실패 처리
    max_retries=2,      # 일시적 네트워크 에러 시 2번까지 재시도
)

# Embedding — 02_build_index와 같은 모델 (반드시 일치해야 함)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large-query")

print("환경 설정 완료 (LLM + Embedding 객체 생성)")

API Key 로드 성공: up_cCu2s...
환경 설정 완료 (LLM + Embedding 객체 생성)


## 2. FAISS 인덱스 로드

02_build_index에서 `../data/faiss_index/`에 저장한 97개 벡터 인덱스를 메모리에 로드.

- `allow_dangerous_deserialization=True`: FAISS는 pickle로 저장·로드하므로 명시적 동의 필요 (본인이 만든 인덱스라 안전)
- `embeddings` 객체를 같이 넘김: 검색 시 질문 임베딩에 필요
- 회귀 검증: 02 결과(목표 50~110개 청크 범위)와 일치하는지 `assert`

In [2]:
from langchain_community.vectorstores import FAISS
from pathlib import Path

INDEX_DIR = "../data/faiss_index"

# allow_dangerous_deserialization=True:
#   FAISS는 pickle 기반 저장/로드라 명시적 동의 필요.
#   본인이 02에서 저장한 인덱스라 안전.
vectorstore = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True,
)

# 회귀 검증: 청크 수 범위 (50~110, 02_build_index 목표 범위)
n_vectors = vectorstore.index.ntotal
assert 50 <= n_vectors <= 110, f"벡터 수 범위 밖: {n_vectors} (기대 50~110)"
print(f"✓ FAISS 인덱스 로드 완료: {n_vectors}개 벡터")
print(f"  로드 경로: {Path(INDEX_DIR).resolve()}")

✓ FAISS 인덱스 로드 완료: 97개 벡터
  로드 경로: C:\Users\wonwo\fastcampus\09.LLM\project\guam-family-chatbot\data\faiss_index


## 3. Retriever 생성 + 검색 동작 빠른 확인

`vectorstore.as_retriever()`로 LCEL 체인에 끼울 수 있는 표준 검색기를 만든다.
`k=3`: 상위 3개 청크 반환 — 02 검증과 동일.

빠른 sanity check: "PIC 리조트 어때?" → 02와 같이 [리조트] PIC 청크가 top 3에 들어오는지.

In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 빠른 검색 동작 확인 (02 결과와 일치하는지)
test_docs = retriever.invoke("PIC 리조트 어때?")
print(f"검색 결과: {len(test_docs)}개\n")

for i, doc in enumerate(test_docs, 1):
    cat = doc.metadata.get("category", "?")
    title = doc.metadata.get("title", "?")
    print(f"[{i}] [{cat}] {title}")
    print(f"    {doc.page_content[:80]}...\n")

검색 결과: 3개

[1] [리조트] 퍼시픽 아일랜드 클럽 (PIC 괌) 소개 - PIC 공식
    # 퍼시픽 아일랜드 클럽 PIC 괌 소개

PIC의 모든 순간, 잊지 못할 특별한 행복의 순간이 됩니다.

## ALL INCLUSIVE RES...

[2] [리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
    1. 개요

Pacific Islands Club, PIC

괌 및 북마리아나 제도에 위치한 미국의 복합형 리조트. 흔히 PIC라는 약칭으로 불...

[3] [액티비티] PIC 괌 워터파크존 시설 안내 - PIC 공식
    # PIC 괌 워터파크 & 액티비티 - 워터파크존

소중한 가족과 함께 신나는 액티비티의 짜릿함을 느껴보세요. PIC 괌 워터파크존에는 가족 단...



## 4. 프롬프트 + RAG 체인 (LCEL)

3개 부품 조립:
- `format_docs`: retriever가 반환한 `list[Document]` → 프롬프트에 들어갈 `string`
- `rag_prompt`: 시스템 메시지(친근 톤 + 출처 인용 + 모르면 모른다) + 휴먼 메시지(질문)
- LCEL 체인: `{context, question} → prompt → llm → StrOutputParser`

체인을 만들기만 하고 invoke는 다음 셀에서 시연 질문으로.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    """retriever 결과(list[Document]) → 프롬프트용 문자열.
    답변에서 출처 인용에 쓸 수 있도록 [category] title 형태로 라벨링."""
    return "\n\n".join(
        f"[{doc.metadata.get('category', '?')}] {doc.metadata.get('title', '?')}\n"
        f"{doc.page_content}"
        for doc in docs
    )


rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 괌 가족여행을 도와주는 친근한 어시스턴트입니다.
주로 부부 + 어린 아이 1~3명이 있는 가족이 질문합니다.

답변 규칙 (규칙 1번이 가장 중요. 다른 규칙과 충돌하면 규칙 1번 우선):

1. 답변의 모든 정보(설명·예시·추천·대안·"~도 좋아요" 같은 사이드 추천 모두 포함)는
   반드시 아래 참고 문서에 명시된 내용이어야 합니다.
   도움이 될 것 같다는 이유로 문서에 없는 시설·장소·서비스(예: 특정 리조트, 식당, 액티비티)를
   답변에 끼워넣지 마세요. 일반 상식·추측·다른 자료 기반 정보 모두 금지.
2. 참고 문서에서 확인할 수 없는 내용은 솔직히 "참고 자료에서 확인되지 않습니다"라고 답하세요.
3. 답변에 사용한 출처를 자연스럽게 인용하세요.
   예: "PIC는 가족 친화 리조트예요 ([리조트] 퍼시픽 아일랜드 클럽 - 나무위키)."
4. 친근한 말투로, 가족 여행자에게 도움 되도록 핵심만 간결하게 정리해주세요.
   단, 규칙 1번을 어기면서까지 도움을 주려 하지 마세요.

참고 문서:
{context}"""),
    ("human", "{question}")
])


# 체인 재구성 (rag_prompt가 바뀌었으니)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✓ RAG 체인 재구성 완료 (프롬프트 강화: 규칙 1번 + 우선순위 명시)")

✓ RAG 체인 재구성 완료 (프롬프트 강화: 규칙 1번 + 우선순위 명시)


## 5. 시연 Q1 검증 — "PIC 리조트 어때? 가족이 가도 좋아?"

**기준**:
> "PIC 리조트 어때?" 같은 질문에 RAG가 **5초 안에 출처와 함께** 답함

확인할 것:
- ✅ 응답 시간 5초 이내
- ✅ 답변에 PIC 정보 포함 (가족 친화·괌 최대 규모·한국어 직원 등)
- ✅ 답변에 출처 인용 (예: `[리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키`)

In [ ]:
import time

question = "PIC 리조트 어때? 가족이 가도 좋아?"

start = time.time()
answer = rag_chain.invoke(question)
elapsed = time.time() - start

print(f"Q: {question}")
print(f"\n⏱️  응답 시간: {elapsed:.2f}초")
print(f"   (기준: 5초 이내)\n")
print("=" * 60)
print("A:")
print(answer)
print("=" * 60)

## 6. 시연 Q2 검증 — "5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?"

**RAG 단독 한계 확인용 셀**.

- 02 검증에서 Climate 청크가 top 3 미진입 (`False`). 검색은 일반 기후 정보 청크는 가져오지만, **시점별("5월 둘째 주") 날씨 정보는 RAG에 애초에 없음** (위키백과는 통시적 기후 통계만 다룸)
- 이 셀의 의도: LLM이 어떻게 답하는지 직접 보고, **도구(`get_guam_weather`)가 필요한 이유를 시연에서 자연스럽게 설명할 근거 확보**
- 시연 #2가 "도구 1개 — 날씨"로 분류 → OpenWeatherMap 도구 추가로 자연 해결 (현재 완료).

In [6]:
import time

question = "5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?"

start = time.time()
answer = rag_chain.invoke(question)
elapsed = time.time() - start

print(f"Q: {question}")
print(f"\n⏱️  응답 시간: {elapsed:.2f}초\n")
print("=" * 60)
print("A:")
print(answer)
print("=" * 60)

# 검색된 청크 — 답변의 근거가 무엇인지 같이 확인
print("\n[검색된 청크 — 답변의 근거]")
docs = retriever.invoke(question)
for i, doc in enumerate(docs, 1):
    cat = doc.metadata.get("category", "?")
    title = doc.metadata.get("title", "?")
    section = doc.metadata.get("section", "")
    label = f"[{cat}] {title}" + (f" / {section}" if section else "")
    print(f"\n[{i}] {label}")
    print(f"    {doc.page_content[:120]}...")

Q: 5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?

⏱️  응답 시간: 2.12초

A:
5월 둘째 주 괌 날씨는 **건기(1월~5월) 말기**로, 대체로 **맑고 건조**한 날씨가 예상됩니다.  
- 평균 최고 기온: 30°C(86°F)  
- 평균 강수량: 3.8~100mm (2월 3.8mm 기록 vs 5월 평균 약 100mm)  
- 우기(7~11월)보다 비가 적지만, 가끔 소나기 가능성 있음  

**우비 필요성**:  
- 대부분 건조하지만, 갑작스러운 소나기 대비해 **휴대용 우비나 우산**을 챙기시는 게 좋아요.  
- 특히 해안가나 산간 지역은 국지성 비가 올 수 있으니 참고하세요.  

> 참고: [괌 (한국어 위키백과)](https://ko.wikipedia.org/wiki/괌) 및 [Guam (English Wikipedia)](https://en.wikipedia.org/wiki/Guam) 기후 자료 기준.  
> 5월은 건기 말이지만, 6월 전환기 전이라 날씨 변동성이 있을 수 있습니다.

[검색된 청크 — 답변의 근거]

[1] [general] 괌 (한국어 위키백과)
    괌에서 가장 높은 산은 람람산, 주물롱망글로산, 볼라노스산, 알마고사산 순이다.

인구

기후

괌의 기후는 열대 우림 기후이지만 낮에는 섭씨 39°C 이상으로 올라가지 않고 밤에도 섭씨 26°C 이하로 내려가지 않...

[2] [general] Guam (English Wikipedia)
    Climate

Guam has a tropical rainforest climate on the Köppen scale (Köppen Af). Its driest month of March almost qualif...

[3] [general] 괌 (한국어 위키백과)
    1898년, 미국이 미국-스페인 전쟁으로 이 섬의 통치권을 차지했다. 마리아나 제도 북부에 있는 섬들을 독일이 거쳐가고 일본이 침략할 동안, 괌은 미국에서 필리핀으로 오고 가는 배들의 정거장

## 7. 결론

### RAG 단독 체크
> "PIC 리조트 어때?" 같은 질문에 RAG가 5초 안에 출처와 함께 답함

- Q1 응답 시간 3.02초 / 출처 인용 / PIC 핵심 정보 풍부 → 통과

### 검증 결과 요약

| 시연 Q | RAG 단독 가능? | 발견 사항 |
|---|---|---|
| Q1 "PIC 리조트 어때?" | ✅ 자료 충실 답변 | 괌·사이판 PIC 정보 혼재 → Agent 페르소나 가드로 해결 |
| Q2 "5월 둘째 주 날씨" | ❌ 시점 정보 자료에 없음 | 환각 발생 → OpenWeatherMap 도구로 자연 해결 |

### 프롬프트 강화 실험 (Q2 기반)

- **변경**: 규칙 1번에 "추천·예시·대안 모두 문서 기반"·"도움 되는 이유로 추가 금지" 명시 + 규칙 1번 우선순위 명시
- **효과 측정**:
  - ✅ "비검색 환각" 차단됨 — 강화 전 Q2 답변 끝에 등장하던 "실내 관광지(예: PIC 리조트)" 사라짐
  - ❌ "시점·수치 환각" 차단 못 함:
    - "월평균 70~100mm" — 자료엔 연평균 2,490mm + 월별 극값(977.6mm / 3.8mm)만
    - "오후/저녁에 소나기 잦음" — 자료에 "afternoon"·"evening"·"오후"·"저녁" 0건 매치
- **결론**: 프롬프트만으론 한계. 강의 S6-1 RAG 한계("맥락 부족 시 환각")와 일치

### 추가 발견 사항 (모두 해결됨)

1. **Q1 "괌·사이판 PIC 혼재"** — Agent 시스템 프롬프트에 "괌만 답변, 사이판 무시" 페르소나 가드로 해결.

2. **Q2 시점·수치 환각** — OpenWeatherMap 도구가 정확한 시점·수치 데이터 제공하므로 자연 해결.

3. **호텔 시설 노후 인지 표현** — 처음에 "노후화" 단어로 grep해 환각 박제로 진단했으나, "오래"로 grep하면 나무위키 PIC에 *"호텔 시설이 오래됐다"* 실제 매칭. 자료 인용 확정 (마지막 셀의 동의어 grep 검사 코드 참조).

In [7]:
import re
from pathlib import Path

PATTERNS = ["오래", "노후", "낡", "구식", "연식", "오랜", "old"]

for fp in sorted(Path("../data/raw").glob("*.md")):
    text = fp.read_text(encoding="utf-8")
    for pat in PATTERNS:
        # 패턴 주변 ±20자 컨텍스트
        for m in re.finditer(pat, text):
            start = max(0, m.start() - 20)
            end = min(len(text), m.end() + 20)
            snippet = text[start:end].replace("\n", " ")
            print(f"[{fp.name}] '{pat}' → ...{snippet}...")

[attractions_Wikivoyage_Regions.md] 'old' → ...The Central Region holds the majority of Gu...
[general_Wikipedia_en.md] 'old' → ...e were 160 Spanish soldiers from Spain and ...
[general_Wikipedia_en.md] 'old' → ...on, Edward Alvarez told the United Nations ...
[general_Wikivoyage_Understand.md] 'old' → ...nd the expansion of older ones. During that...
[resorts_PIC_goldcard_picofficial.md] 'old' → ....pic.co.kr/content/goldcard" category: "리조트...
[resorts_PIC_namuwiki.md] '오래' → ...구성되어 있다.  다만 호텔 시설이 오래됐다는 단점이 있다, 다만 호텔의 서...
[transport_Wikivoyage_Get_around.md] 'old' → ...am Bombs" which are older vehicles that are...
[transport_Wikivoyage_Get_around.md] 'old' → ...onally, many of the older roads were paved ...
[transport_Wikivoyage_Get_around.md] 'old' → ...ances between households, and to small vill...
